# FireSight-IR | Module 3b — Analysis & Ablation Study

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Google Colab free tier (T4 GPU) — Runtime → Change runtime type → T4 GPU  
**Depends on:** Module 3a — `firesight_pinn_best.pt` must exist on Drive

---

## Module overview

| Section | Content | Runtime |
|---|---|---|
| 0 | Environment + Drive mount | 1 min |
| 1 | Configuration & paths | — |
| 2 | Shared definitions (model, loss, dataset) | — |
| 3 | Load full model + collect baseline predictions | 3 min |
| **A** | **Probability score distributions** (wildfire vs false-alarm) | 2 min |
| **B** | **Ablation study** — full retrain of 3 stripped variants (30 epochs each) | ~6 hrs |
| **C** | **Extended metrics** — wildfire precision, annotated confusion matrices | 5 min |
| **D** | **Ablation convergence plots** — loss/accuracy curves per variant | 2 min |

### Reconnection safety
Every ablation variant saves its **own checkpoint** and **training log** after each epoch.  
If Colab disconnects, re-run **Sections 0–3 only**, then re-run whichever Section B cell was interrupted.  
Training resumes automatically from the last saved epoch.

### Output files
All figures saved to `{BASE_DIR}/figures/03b/` at 200 DPI.  
All ablation checkpoints and logs saved to `{BASE_DIR}/models/ablations/`.


---
## Section 0 — Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install torch tqdm scikit-learn matplotlib numpy pandas h5py pyarrow scipy -q

import os, json, time, copy, warnings
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_fscore_support, average_precision_score
)
from tqdm.auto import tqdm
import numpy._core.multiarray
torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])
warnings.filterwarnings('ignore')

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_AMP = device.type == 'cuda'

print(f'Device : {device}')
if HAS_AMP:
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'AMP    : enabled (FP16)')
else:
    print('[STOP] No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')
print(f'PyTorch: {torch.__version__}')

---
## Section 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR      = '/content/drive/MyDrive/firesight-ir'
ARCHIVE_PATH  = f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
SPLIT_DIR     = f'{BASE_DIR}/data/splits'
MODEL_DIR     = f'{BASE_DIR}/models'
ABLATION_DIR  = f'{BASE_DIR}/models/ablations'
FIGURES_DIR   = f'{BASE_DIR}/figures/03b'
CACHE_DIR     = f'{BASE_DIR}/data/cache'

WEIGHTS_PATH  = f'{BASE_DIR}/data/scalers/class_weights_v2.json'
if not os.path.exists(WEIGHTS_PATH):
    WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights.json'
    print('[WARN] Using v1 class weights.')

BEST_PATH     = f'{MODEL_DIR}/firesight_pinn_best.pt'

for d in [ABLATION_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Feature dimensions ────────────────────────────────────────────────────────
N_ATM=16; N_SRF=20; N_DERIVED=6; PATCH_SIZE=32; N_CHANNELS=4; N_CLASSES=3

# ── Training hyperparameters (must match Module 3a) ───────────────────────────
BATCH_SIZE=1024; N_EPOCHS=30; LR_MAX=3.5e-4; WEIGHT_DECAY=1e-4; DROPOUT=0.3
NUM_WORKERS=2
LAMBDA_BL=0.10; LAMBDA_DR=0.05; LAMBDA_TH=0.05
BT_I4_FIRE_MIN=310.0; BTD_FIRE_MIN=10.0

SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
if HAS_AMP: torch.cuda.manual_seed(SEED)

# ── Cache file map ────────────────────────────────────────────────────────────
CACHE_FILES = {
    'patches': f'{CACHE_DIR}/patches.npy',
    'atm'    : f'{CACHE_DIR}/atm.npy',
    'srf'    : f'{CACHE_DIR}/srf.npy',
    'derived': f'{CACHE_DIR}/derived.npy',
    'labels' : f'{CACHE_DIR}/labels.npy',
    'aux'    : f'{CACHE_DIR}/aux.npy',
}

# ── Plot theme ────────────────────────────────────────────────────────────────
BG     = '#0D1117'
PANEL  = '#161B22'
PANEL2 = '#1C2128'
TEXT   = '#E6EDF3'
SUB    = '#8B949E'
BORDER = '#30363D'
ACCENT = '#388BFD'

CPAL   = {'wildfire':'#E85D04', 'false-alarm':'#D97706', 'non-fire':'#3A5C8A'}
APAL   = {  # ablation variant colours
    'Full model'      : '#4FC3F7',
    'No physics loss' : '#F78166',
    'No ERA5 (ATM)'   : '#FFA657',
    'No surface (SRF)': '#7EE787',
}

plt.rcParams.update({
    'figure.facecolor' : BG,
    'axes.facecolor'   : PANEL,
    'axes.edgecolor'   : BORDER,
    'text.color'       : TEXT,
    'xtick.color'      : SUB,
    'ytick.color'      : SUB,
    'axes.labelcolor'  : SUB,
    'grid.color'       : BORDER,
    'font.family'      : 'DejaVu Sans',
    'axes.titlepad'    : 10,
})

print('Configuration set.')
print(f'  Figures → {FIGURES_DIR}')
print(f'  Ablation checkpoints → {ABLATION_DIR}')

---
## Section 2 — Shared definitions

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Dataset
# ═════════════════════════════════════════════════════════════════════════════
class FireSightDataset(Dataset):
    def __init__(self, cache_files, indices, augment=False,
                 zero_atm=False, zero_srf=False, zero_der=False):
        self.indices  = np.sort(indices)
        self.augment  = augment
        self.zero_atm = zero_atm
        self.zero_srf = zero_srf
        self.zero_der = zero_der
        self.patches  = np.load(cache_files['patches'], mmap_mode='r')
        self.atm      = np.load(cache_files['atm'],     mmap_mode='r')
        self.srf      = np.load(cache_files['srf'],     mmap_mode='r')
        self.derived  = np.load(cache_files['derived'], mmap_mode='r')
        self.labels   = np.load(cache_files['labels'],  mmap_mode='r')
        self.aux      = np.load(cache_files['aux'],     mmap_mode='r')

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx   = int(self.indices[i])
        patch = self.patches[idx].copy()
        if self.augment:
            k = np.random.randint(0, 4)
            if k: patch = np.rot90(patch, k, axes=(1, 2)).copy()
            if np.random.rand() > 0.5: patch = np.flip(patch, axis=2).copy()
        atm = np.zeros(N_ATM,    dtype=np.float32) if self.zero_atm else self.atm[idx].copy()
        srf = np.zeros(N_SRF,    dtype=np.float32) if self.zero_srf else self.srf[idx].copy()
        der = np.zeros(N_DERIVED, dtype=np.float32) if self.zero_der else self.derived[idx].copy()
        return (
            torch.from_numpy(patch),
            torch.from_numpy(atm),
            torch.from_numpy(srf),
            torch.from_numpy(der),
            torch.tensor(int(self.labels[idx]), dtype=torch.long),
            torch.from_numpy(self.aux[idx].copy()),
        )


# ═════════════════════════════════════════════════════════════════════════════
# Model
# ═════════════════════════════════════════════════════════════════════════════
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim))
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    def forward(self, x): return F.relu(self.net(x) + self.proj(x))

class CNNBranch(nn.Module):
    def __init__(self, in_ch=4, dropout=0.2):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(in_ch,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32,32,3,padding=1),    nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),    nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64,64,3,padding=1),    nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),   nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2))
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x): return self.drop(self.gap(self.enc(x)).flatten(1))

class FireSightPINN(nn.Module):
    """
    Multi-branch PINN:
      CNN(128) + MLP-atm(32) + MLP-srf(32) + MLP-der(16) = 208 → 3-class
      + transmittance head for Beer-Lambert physics constraint.
    """
    def __init__(self, n_atm=16, n_srf=20, n_der=6, n_cls=3, drop=0.3):
        super().__init__()
        self.cnn    = CNNBranch(4, drop)
        self.atm    = nn.Sequential(ResidualBlock(n_atm,64), ResidualBlock(64,32))
        self.srf    = nn.Sequential(ResidualBlock(n_srf,64), ResidualBlock(64,32))
        self.der    = nn.Sequential(
            nn.Linear(n_der,32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32,16),    nn.BatchNorm1d(16), nn.ReLU())
        self.fusion = nn.Sequential(
            nn.Linear(208,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128,64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(drop))
        self.cls    = nn.Linear(64, n_cls)
        self.trans  = nn.Sequential(nn.Linear(64,16), nn.ReLU(), nn.Linear(16,1), nn.Sigmoid())

    def forward(self, patch, atm, srf, der):
        feat = self.fusion(
            torch.cat([self.cnn(patch), self.atm(atm), self.srf(srf), self.der(der)], dim=1))
        return self.cls(feat), self.trans(feat)


# ═════════════════════════════════════════════════════════════════════════════
# Loss functions
# ═════════════════════════════════════════════════════════════════════════════
class PINNLoss(nn.Module):
    def __init__(self, weights, lbl=0.1, ldr=0.05, lth=0.05,
                 bt_min=310., btd_min=10., device='cpu'):
        super().__init__()
        self.lbl, self.ldr, self.lth = lbl, ldr, lth
        self.bt_min, self.btd_min = bt_min, btd_min
        w = torch.tensor(weights, dtype=torch.float32).to(device)
        self.ce  = nn.CrossEntropyLoss(weight=w)
        self.mse = nn.MSELoss()
    def forward(self, logits, trans, labels, aux):
        L_ce = self.ce(logits, labels)
        L_bl = self.mse(trans, aux[:, 2:3])
        p_wf = F.softmax(logits, dim=1)[:, 1]
        L_dr = (p_wf * (aux[:, 0] < self.bt_min).float()).mean()
        L_th = (p_wf * (aux[:, 1] < self.btd_min).float()).mean()
        total = L_ce + self.lbl*L_bl + self.ldr*L_dr + self.lth*L_th
        return total, {'ce': L_ce.item(), 'bl': L_bl.item(),
                       'dr': L_dr.item(), 'th': L_th.item()}

class CEOnlyLoss(nn.Module):
    """Cross-entropy only — no physics terms. Used for no-physics ablation."""
    def __init__(self, weights, device='cpu'):
        super().__init__()
        w = torch.tensor(weights, dtype=torch.float32).to(device)
        self.ce = nn.CrossEntropyLoss(weight=w)
    def forward(self, logits, trans, labels, aux):
        loss = self.ce(logits, labels)
        return loss, {'ce': loss.item(), 'bl': 0., 'dr': 0., 'th': 0.}


# ═════════════════════════════════════════════════════════════════════════════
# Shared utilities
# ═════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def collect_outputs(model, loader):
    """Run inference on a loader. Returns (preds, labels, probs)."""
    model.eval()
    preds_l, labels_l, probs_l = [], [], []
    for patch, atm, srf, der, labels, aux in loader:
        with autocast(enabled=HAS_AMP):
            logits, _ = model(
                patch.to(device), atm.to(device),
                srf.to(device),   der.to(device))
        probs_l.append(F.softmax(logits, dim=1).cpu().numpy())
        preds_l.append(logits.argmax(1).cpu().numpy())
        labels_l.append(labels.numpy())
    return (np.concatenate(preds_l),
            np.concatenate(labels_l),
            np.concatenate(probs_l))


def compute_metrics(preds, labels, probs, variant_name):
    """Compute full metric dict for one variant."""
    prec, rec, f1, sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0,1,2], zero_division=0)
    fa_auc = roc_auc_score((labels==2).astype(int), probs[:,2]) \
        if (labels==2).sum() > 0 else 0.0
    wf_auc = roc_auc_score((labels==1).astype(int), probs[:,1])
    overall_acc = float((preds == labels).mean())
    return {
        'variant'      : variant_name,
        'val_acc'      : overall_acc,
        'nf_precision' : float(prec[0]), 'nf_recall': float(rec[0]), 'nf_f1': float(f1[0]),
        'wf_precision' : float(prec[1]), 'wf_recall': float(rec[1]), 'wf_f1': float(f1[1]),
        'fa_precision' : float(prec[2]), 'fa_recall': float(rec[2]), 'fa_f1': float(f1[2]),
        'fa_auc'       : float(fa_auc),
        'wf_auc'       : float(wf_auc),
    }


def savefig(name):
    """Save current figure to FIGURES_DIR at 200 DPI."""
    path = f'{FIGURES_DIR}/{name}'
    plt.savefig(path, dpi=200, bbox_inches='tight', facecolor=BG)
    print(f'  Saved → {path}')
    plt.show()
    plt.close()


print('All shared definitions loaded.')

---
## Section 3 — Load full model and baseline predictions

In [ ]:
# ── Splits ────────────────────────────────────────────────────────────────────
train_idx = np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx   = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx  = np.load(f'{SPLIT_DIR}/test_index.npy')
print(f'Splits  Train:{len(train_idx):,}  Val:{len(val_idx):,}  Test:{len(test_idx):,}')

# ── Class weights ─────────────────────────────────────────────────────────────
with open(WEIGHTS_PATH) as f: cw = json.load(f)
if   isinstance(cw, list): class_weights = cw
elif isinstance(cw, dict): class_weights = cw.get('_list', [6.9, 0.36, 9.81])
else:                      class_weights = [6.9, 0.36, 9.81]
print(f'Class weights: {[round(w,3) for w in class_weights]}')

# ── DataLoaders ───────────────────────────────────────────────────────────────
pin = device.type == 'cuda'
lkw = dict(num_workers=NUM_WORKERS, pin_memory=pin,
           prefetch_factor=2 if NUM_WORKERS > 0 else None,
           persistent_workers=False)

train_ds = FireSightDataset(CACHE_FILES, train_idx, augment=True)
val_ds   = FireSightDataset(CACHE_FILES, val_idx,   augment=False)
test_ds  = FireSightDataset(CACHE_FILES, test_idx,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,  **lkw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, **lkw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, **lkw)
print(f'Loaders ready. Train batches: {len(train_loader):,}')

In [ ]:
# ── Load full model checkpoint ────────────────────────────────────────────────
assert os.path.exists(BEST_PATH), \
    f'Checkpoint not found: {BEST_PATH}\nRun Module 3a training first.'

full_model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
full_model.load_state_dict(ckpt['model_state_dict'])
full_model.eval()
n_params = sum(p.numel() for p in full_model.parameters())
print(f'Full model loaded')
print(f'  Epoch      : {ckpt["epoch"]}')
print(f'  Val loss   : {ckpt["val_loss"]:.4f}')
print(f'  Parameters : {n_params:,}')

# ── Collect baseline predictions on val and test ──────────────────────────────
print('\nCollecting val predictions...')
val_preds, val_labels, val_probs = collect_outputs(full_model, val_loader)
print('Collecting test predictions...')
test_preds, test_labels, test_probs = collect_outputs(full_model, test_loader)

full_val_metrics  = compute_metrics(val_preds,  val_labels,  val_probs,  'Full model')
full_test_metrics = compute_metrics(test_preds, test_labels, test_probs, 'Full model')

print(f'\nFull model — Val 2023')
print(f'  Overall acc    : {full_val_metrics["val_acc"]:.4f}')
print(f'  WF recall      : {full_val_metrics["wf_recall"]:.4f}')
print(f'  WF precision   : {full_val_metrics["wf_precision"]:.4f}')
print(f'  FA recall      : {full_val_metrics["fa_recall"]:.4f}')
print(f'  FA AUC         : {full_val_metrics["fa_auc"]:.4f}')

---
## Section A — Probability Score Distributions

Plots the model's wildfire probability score P(wildfire) for each ground-truth class.  
The non-overlap between wildfire and false-alarm distributions is the geometric proof of AUC = 1.0000.

In [ ]:
p_wf_wf = val_probs[val_labels == 1, 1]   # P(wildfire) | true wildfire
p_wf_fa = val_probs[val_labels == 2, 1]   # P(wildfire) | true false-alarm
p_wf_nf = val_probs[val_labels == 0, 1]   # P(wildfire) | true non-fire

fa_leak  = (p_wf_fa > 0.5).mean() * 100
wf_miss  = (p_wf_wf < 0.5).mean() * 100

print(f'P(wildfire) statistics — Val 2023')
print(f'  True wildfire    median={np.median(p_wf_wf):.4f}  n={len(p_wf_wf):,}')
print(f'  True false-alarm median={np.median(p_wf_fa):.4f}  n={len(p_wf_fa):,}')
print(f'  True non-fire    median={np.median(p_wf_nf):.4f}  n={len(p_wf_nf):,}')
print(f'\n  FA leakage  (P(wf)>0.5): {fa_leak:.3f}%')
print(f'  WF miss rate (P(wf)<0.5): {wf_miss:.3f}%')

In [ ]:
BINS = np.linspace(0, 1, 61)

fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

# ── A1: Wildfire vs False-alarm (main result) ─────────────────────────────────
ax = fig.add_subplot(gs[0, :2])
ax.set_facecolor(PANEL)
ax.hist(p_wf_wf, bins=BINS, color=CPAL['wildfire'],    alpha=0.80, density=True,
        label=f'True wildfire   n={len(p_wf_wf):,}  median={np.median(p_wf_wf):.3f}')
ax.hist(p_wf_fa, bins=BINS, color=CPAL['false-alarm'], alpha=0.80, density=True,
        label=f'True false-alarm  n={len(p_wf_fa):,}  median={np.median(p_wf_fa):.3f}')
ax.axvline(0.5, color='white', lw=1.8, ls='--', alpha=0.7, label='Decision threshold = 0.5')
ax.axvline(np.median(p_wf_wf), color=CPAL['wildfire'],    lw=1.2, ls=':')
ax.axvline(np.median(p_wf_fa), color=CPAL['false-alarm'], lw=1.2, ls=':')

# Annotate leakage
if fa_leak > 0:
    ax.text(0.52, ax.get_ylim()[1]*0.85 if ax.get_ylim()[1]>0 else 5,
            f'FA leakage: {fa_leak:.2f}%', color=CPAL['false-alarm'],
            fontsize=9, va='top')
else:
    ax.text(0.52, 0.95, 'FA leakage: 0.00%\n(AUC = 1.0000)',
            color='#7EE787', fontsize=10, va='top',
            transform=ax.transAxes, fontweight='bold')

ax.set_xlabel('P(wildfire)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Wildfire vs False-Alarm  —  P(wildfire) score distributions\nVal 2023 (fully held-out)',
             color=TEXT, fontsize=12, fontweight='bold')
ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.grid(alpha=0.18); ax.spines[['top','right']].set_visible(False)

# ── A2: All three classes ─────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
ax.set_facecolor(PANEL)
for arr, key, lbl in [
    (p_wf_wf, 'wildfire',    f'Wildfire  (n={len(p_wf_wf):,})'),
    (p_wf_fa, 'false-alarm', f'False-alarm (n={len(p_wf_fa):,})'),
    (p_wf_nf, 'non-fire',    f'Non-fire  (n={len(p_wf_nf):,})'),
]:
    ax.hist(arr, bins=BINS, color=CPAL[key], alpha=0.72, density=True, label=lbl)
ax.axvline(0.5, color='white', lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('P(wildfire)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('All classes', color=TEXT, fontsize=11, fontweight='bold')
ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax.grid(alpha=0.18); ax.spines[['top','right']].set_visible(False)

# ── A3: Log-scale zoom on false-alarm scores ──────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
ax.set_facecolor(PANEL)
ax.hist(p_wf_fa, bins=BINS, color=CPAL['false-alarm'], alpha=0.85, density=True)
ax.set_yscale('log')
ax.axvline(0.5, color='white', lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('P(wildfire)', fontsize=10)
ax.set_ylabel('Density (log)', fontsize=10)
ax.set_title('False-alarm scores — log scale\n(reveals tail behaviour)', color=TEXT, fontsize=10)
ax.grid(alpha=0.18); ax.spines[['top','right']].set_visible(False)

# ── A4: CDF comparison ────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
ax.set_facecolor(PANEL)
for arr, key, lbl in [
    (p_wf_wf, 'wildfire',    'Wildfire'),
    (p_wf_fa, 'false-alarm', 'False-alarm'),
    (p_wf_nf, 'non-fire',    'Non-fire'),
]:
    sorted_arr = np.sort(arr)
    cdf = np.arange(1, len(sorted_arr)+1) / len(sorted_arr)
    ax.plot(sorted_arr, cdf, color=CPAL[key], lw=2, label=lbl)
ax.axvline(0.5, color='white', lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('P(wildfire)', fontsize=10)
ax.set_ylabel('Cumulative fraction', fontsize=10)
ax.set_title('CDF — P(wildfire) by class', color=TEXT, fontsize=10)
ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax.grid(alpha=0.18); ax.spines[['top','right']].set_visible(False)

# ── A5: Summary statistics table ──────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
ax.set_facecolor(PANEL2)
ax.axis('off')
stats_data = [
    ['Metric', 'Wildfire', 'False-alarm', 'Non-fire'],
    ['n', f'{len(p_wf_wf):,}', f'{len(p_wf_fa):,}', f'{len(p_wf_nf):,}'],
    ['Median P(wf)', f'{np.median(p_wf_wf):.4f}', f'{np.median(p_wf_fa):.4f}', f'{np.median(p_wf_nf):.4f}'],
    ['Mean P(wf)',   f'{np.mean(p_wf_wf):.4f}',   f'{np.mean(p_wf_fa):.4f}',   f'{np.mean(p_wf_nf):.4f}'],
    ['P10',  f'{np.percentile(p_wf_wf,10):.4f}', f'{np.percentile(p_wf_fa,10):.4f}', f'{np.percentile(p_wf_nf,10):.4f}'],
    ['P90',  f'{np.percentile(p_wf_wf,90):.4f}', f'{np.percentile(p_wf_fa,90):.4f}', f'{np.percentile(p_wf_nf,90):.4f}'],
    ['P(wf)>0.5', f'{(p_wf_wf>0.5).mean()*100:.2f}%',
                  f'{(p_wf_fa>0.5).mean()*100:.2f}%',
                  f'{(p_wf_nf>0.5).mean()*100:.2f}%'],
    ['FA AUC',    f'{full_val_metrics["fa_auc"]:.4f}', '—', '—'],
    ['WF AUC',    f'{full_val_metrics["wf_auc"]:.4f}', '—', '—'],
]
tbl = ax.table(cellText=stats_data[1:], colLabels=stats_data[0],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r,c), cell in tbl.get_celld().items():
    cell.set_facecolor(PANEL if r%2==0 else PANEL2)
    cell.set_edgecolor(BORDER)
    cell.set_text_props(color=TEXT)
    if r == 0:
        cell.set_facecolor('#2D333B')
        cell.set_text_props(color=TEXT, fontweight='bold')
ax.set_title('P(wildfire) summary statistics — Val 2023', color=TEXT, fontsize=10, pad=8)

fig.suptitle('FireSight-IR  |  Module 3b — Section A: Probability Score Distributions',
             color=TEXT, fontsize=14, fontweight='bold', y=1.01)
savefig('A_score_distributions.png')

---
## Section B — Ablation Study (full 30-epoch retraining)

Three variants are retrained from scratch with one component removed per run:

| Variant | What is removed | Expected impact |
|---|---|---|
| **No physics loss** | BL + DR + TH loss terms set to 0 | FA AUC degrades; WF/FA separation narrows |
| **No ERA5 (ATM)** | ATM branch inputs zeroed during training | Atmospheric correction lost |
| **No surface (SRF)** | SRF branch inputs zeroed during training | False-alarm suppression weakened |

Each variant:
- Saves a checkpoint after **every epoch** to `models/ablations/`
- Saves a training log after every epoch
- **Auto-resumes** from the last saved epoch if Colab disconnects
- Runs the same 30 epochs as the full model for fair comparison

**Total runtime on T4:** ~6 hours (2 hrs/variant × 3 variants)

In [ ]:
def run_ablation_variant(
    variant_name,
    zero_atm=False, zero_srf=False, zero_der=False,
    use_physics=True,
    n_epochs=N_EPOCHS
):
    """
    Train one ablation variant with full resume support.
    Saves checkpoint + log after every epoch.
    Returns final val metric dict.
    """
    slug         = variant_name.lower().replace(' ', '_').replace('(','').replace(')','').replace('/','_')
    ckpt_path    = f'{ABLATION_DIR}/{slug}_best.pt'
    latest_path  = f'{ABLATION_DIR}/{slug}_latest.pt'
    log_path     = f'{ABLATION_DIR}/{slug}_log.json'
    metrics_path = f'{ABLATION_DIR}/{slug}_metrics.json'

    # ── If already completed, load saved metrics and return ───────────────────
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            saved = json.load(f)
        if saved.get('completed', False):
            print(f'[SKIP] {variant_name} already complete. Loading saved metrics.')
            return saved

    print(f'\n{"═"*62}')
    print(f'  Ablation variant : {variant_name}')
    print(f'  zero_atm={zero_atm}  zero_srf={zero_srf}  zero_der={zero_der}')
    print(f'  use_physics={use_physics}  epochs={n_epochs}')
    print(f'  Checkpoint  → {ckpt_path}')
    print(f'  Log         → {log_path}')
    print(f'{"═"*62}')

    torch.manual_seed(SEED); np.random.seed(SEED)
    if HAS_AMP: torch.cuda.manual_seed(SEED)

    # ── Datasets and loaders ─────────────────────────────────────────────────
    a_train_ds = FireSightDataset(
        CACHE_FILES, train_idx, augment=True,
        zero_atm=zero_atm, zero_srf=zero_srf, zero_der=zero_der)
    a_val_ds   = FireSightDataset(
        CACHE_FILES, val_idx, augment=False,
        zero_atm=zero_atm, zero_srf=zero_srf, zero_der=zero_der)

    a_train_loader = DataLoader(a_train_ds, batch_size=BATCH_SIZE,   shuffle=True,  **lkw)
    a_val_loader   = DataLoader(a_val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, **lkw)

    # ── Model and loss ────────────────────────────────────────────────────────
    model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)

    criterion = PINNLoss(
        class_weights, LAMBDA_BL, LAMBDA_DR, LAMBDA_TH,
        BT_I4_FIRE_MIN, BTD_FIRE_MIN, device
    ) if use_physics else CEOnlyLoss(class_weights, device)

    scaler = GradScaler(enabled=HAS_AMP)

    best_val_loss = float('inf')
    start_epoch   = 0
    training_log  = []

    # ── Resume from latest checkpoint if available ────────────────────────────
    resume_path = latest_path if os.path.exists(latest_path) else None
    if resume_path:
        try:
            saved_ckpt = torch.load(resume_path, map_location=device, weights_only=False)
            model.load_state_dict(saved_ckpt['model_state_dict'])
            start_epoch   = saved_ckpt.get('epoch', 0)
            best_val_loss = saved_ckpt.get('best_val_loss', float('inf'))
            if os.path.exists(log_path):
                with open(log_path) as f: training_log = json.load(f)
            print(f'  ✓ Resumed from epoch {start_epoch} | best_val_loss={best_val_loss:.4f}')
            print(f'    Epochs remaining: {n_epochs - start_epoch}')
        except Exception as e:
            print(f'  [WARN] Resume failed ({e}). Starting fresh.')
            start_epoch = 0; training_log = []
    else:
        print(f'  Starting fresh (epoch 1/{n_epochs})')

    if start_epoch >= n_epochs:
        print(f'  ✓ Already trained {n_epochs} epochs. Loading best checkpoint for eval.')
    else:
        optimizer = AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
        if resume_path:
            try:
                opt_ckpt = torch.load(resume_path, map_location=device, weights_only=False)
                if 'optimizer_state' in opt_ckpt:
                    optimizer.load_state_dict(opt_ckpt['optimizer_state'])
            except: pass

        completed_steps = start_epoch * len(a_train_loader)
        scheduler = OneCycleLR(
            optimizer, max_lr=LR_MAX,
            steps_per_epoch=len(a_train_loader), epochs=n_epochs,
            pct_start=0.1, anneal_strategy='cos',
            last_epoch=completed_steps - 1)

        est_min = len(a_train_loader) * 9 // 60
        print(f'\n  Training epochs {start_epoch+1}–{n_epochs}')
        print(f'  Estimated ~{est_min}–{est_min+2} min/epoch on T4')
        print(f'  Progress autosaved every epoch — safe to disconnect\n')

        for epoch in range(start_epoch, n_epochs):
            t0 = time.time()

            # ── Train ─────────────────────────────────────────────────────────
            model.train()
            total_loss = 0
            comp_acc   = {'ce':0,'bl':0,'dr':0,'th':0}
            correct = n_total = 0

            bar = tqdm(a_train_loader,
                       desc=f'  [{variant_name}] Ep {epoch+1:02d}/{n_epochs}',
                       ncols=115, leave=True)
            for patch, atm, srf, der, labels, aux in bar:
                patch  = patch.to(device,  non_blocking=True)
                atm    = atm.to(device,    non_blocking=True)
                srf    = srf.to(device,    non_blocking=True)
                der    = der.to(device,    non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                aux    = aux.to(device,    non_blocking=True)

                optimizer.zero_grad()
                with autocast(enabled=HAS_AMP):
                    logits, trans = model(patch, atm, srf, der)
                    loss, comp    = criterion(logits, trans, labels, aux)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()

                bs = len(labels); total_loss += loss.item() * bs
                for k in comp_acc: comp_acc[k] += comp[k] * bs
                correct += (logits.argmax(1) == labels).sum().item()
                n_total += bs
                bar.set_postfix(
                    loss=f'{loss.item():.3f}',
                    acc =f'{correct/n_total:.3f}',
                    lr  =f'{scheduler.get_last_lr()[0]:.1e}')

            train_loss = total_loss / n_total
            train_acc  = correct / n_total
            avg_comp   = {k: v/n_total for k,v in comp_acc.items()}

            # ── Validate ──────────────────────────────────────────────────────
            model.eval()
            val_total = 0; vp, vl = [], []
            with torch.no_grad():
                for patch, atm, srf, der, labels, aux in a_val_loader:
                    with autocast(enabled=HAS_AMP):
                        logits, trans = model(
                            patch.to(device), atm.to(device),
                            srf.to(device),   der.to(device))
                        v_loss, _ = criterion(
                            logits, trans, labels.to(device), aux.to(device))
                    val_total += v_loss.item() * len(labels)
                    vp.append(logits.argmax(1).cpu().numpy())
                    vl.append(labels.numpy())

            val_loss   = val_total / len(val_idx)
            vp_arr     = np.concatenate(vp)
            vl_arr     = np.concatenate(vl)
            val_acc    = (vp_arr == vl_arr).mean()
            pca = {}
            for cls, nm in [(0,'nf'),(1,'wf'),(2,'fa')]:
                m = vl_arr == cls
                pca[nm] = float((vp_arr[m] == cls).mean()) if m.sum() > 0 else 0.
            elapsed = time.time() - t0

            print(f'  → val={val_loss:.4f} acc={val_acc:.3f} | '
                  f'nf={pca["nf"]:.3f} wf={pca["wf"]:.3f} fa={pca["fa"]:.3f} | '
                  f'{elapsed/60:.1f} min')
            print(f'     CE={avg_comp["ce"]:.4f} BL={avg_comp["bl"]:.4f} '
                  f'DR={avg_comp["dr"]:.4f} TH={avg_comp["th"]:.4f}')

            epoch_log = {
                'epoch'      : epoch + 1,
                'train_loss' : round(train_loss, 4),
                'train_acc'  : round(train_acc,  4),
                'val_loss'   : round(val_loss,   4),
                'val_acc'    : round(val_acc,     4),
                'elapsed_s'  : round(elapsed,     1),
                'loss_components'   : {k: round(v,5) for k,v in avg_comp.items()},
                'per_class_val_acc' : {k: round(v,4) for k,v in pca.items()},
            }
            training_log.append(epoch_log)

            # ── Save latest (for resume) + best ──────────────────────────────
            latest_state = {
                'epoch'            : epoch + 1,
                'model_state_dict' : model.state_dict(),
                'optimizer_state'  : optimizer.state_dict(),
                'best_val_loss'    : min(best_val_loss, val_loss),
                'val_loss'         : val_loss,
                'val_acc'          : val_acc,
            }
            torch.save(latest_state, latest_path)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(latest_state, ckpt_path)
                print(f'     ✓ Best saved (val_loss={val_loss:.4f})')

            with open(log_path, 'w') as f:
                json.dump(training_log, f, indent=2)

    # ── Final evaluation on best checkpoint ───────────────────────────────────
    best_ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])

    final_preds, final_labels, final_probs = collect_outputs(model, a_val_loader)
    metrics = compute_metrics(final_preds, final_labels, final_probs, variant_name)
    metrics['best_val_loss'] = float(best_ckpt.get('val_loss', best_val_loss))
    metrics['best_epoch']    = int(best_ckpt.get('epoch', n_epochs))
    metrics['log_path']      = log_path
    metrics['completed']     = True

    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f'\n  ✓ {variant_name} complete')
    print(f'    Best epoch={metrics["best_epoch"]}  val_loss={metrics["best_val_loss"]:.4f}')
    print(f'    WF recall={metrics["wf_recall"]:.4f}  '
          f'WF precision={metrics["wf_precision"]:.4f}  '
          f'FA AUC={metrics["fa_auc"]:.4f}')
    return metrics


print('Ablation framework ready.')
print('Run each variant cell below independently.')
print('Each auto-resumes if Colab disconnects.')

In [ ]:
# ── Variant 1: No physics loss ────────────────────────────────────────────────
# Removes Beer-Lambert, dynamic range, and thermal realism loss terms.
# Tests how much the physics constraints contribute to FA separation.
# Runtime: ~2 hrs on T4  |  Auto-resumes if disconnected

metrics_no_physics = run_ablation_variant(
    'No physics loss',
    zero_atm=False, zero_srf=False, zero_der=False,
    use_physics=False
)

In [ ]:
# ── Variant 2: No ERA5 / ATM branch ──────────────────────────────────────────
# ATM branch receives all-zeros throughout training and inference.
# Tests how much ERA5 atmospheric context contributes.
# Runtime: ~2 hrs on T4  |  Auto-resumes if disconnected

metrics_no_atm = run_ablation_variant(
    'No ERA5 (ATM)',
    zero_atm=True, zero_srf=False, zero_der=False,
    use_physics=True
)

In [ ]:
# ── Variant 3: No surface / SRF branch ───────────────────────────────────────
# SRF branch receives all-zeros throughout training and inference.
# Tests how much OSM/land cover context contributes to FA suppression.
# Runtime: ~2 hrs on T4  |  Auto-resumes if disconnected

metrics_no_srf = run_ablation_variant(
    'No surface (SRF)',
    zero_atm=False, zero_srf=True, zero_der=False,
    use_physics=True
)

In [ ]:
# ── Assemble all results ──────────────────────────────────────────────────────
# Add full model metrics (from Section 3, no retraining needed)
full_val_metrics['best_val_loss'] = float(ckpt['val_loss'])
full_val_metrics['best_epoch']    = int(ckpt['epoch'])

ablation_results = [
    full_val_metrics,
    metrics_no_physics,
    metrics_no_atm,
    metrics_no_srf,
]

all_results_path = f'{ABLATION_DIR}/all_ablation_results.json'
with open(all_results_path, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'All ablation results saved → {all_results_path}')

# ── Delta table ───────────────────────────────────────────────────────────────
base = ablation_results[0]
keys = ['wf_recall','wf_precision','wf_f1','fa_recall','fa_auc','val_acc']
hdr  = f'{"Variant":<22}' + ''.join(f'{k:>14}' for k in keys)
print(f'\nDelta from full model (negative = degradation):')
print(hdr)
print('-' * (22 + 14*len(keys)))
for r in ablation_results:
    row = f'{r["variant"]:<22}'
    for k in keys:
        d = r[k] - base[k]
        row += f'{"+" if d>=0 else ""}{d:+.4f}    '
    print(row)

---
## Section C — Extended Metrics

Full classification report with wildfire precision highlighted, plus annotated confusion matrices showing both absolute counts with row percentages and normalised rates.

In [ ]:
CNAMES = ['Non-fire', 'Wildfire', 'False-alarm']

def print_extended_report(preds, labels, probs, split_name):
    print(f'\n{"═"*62}')
    print(f'  {split_name}')
    print(f'{"═"*62}')
    print(classification_report(labels, preds, target_names=CNAMES, digits=4))
    prec, rec, f1, sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0,1,2], zero_division=0)
    print('Key wildfire metrics:')
    print(f'  Precision : {prec[1]:.4f}')
    print(f'    → of {(preds==1).sum():,} pixels predicted wildfire, '
          f'{int(prec[1]*(preds==1).sum()):,} are genuinely wildfire')
    print(f'  Recall    : {rec[1]:.4f}')
    print(f'    → of {(labels==1).sum():,} true wildfire pixels, '
          f'{int(rec[1]*(labels==1).sum()):,} are correctly detected')
    print(f'  F1        : {f1[1]:.4f}')
    if (labels==2).sum() > 0:
        fa_auc = roc_auc_score((labels==2).astype(int), probs[:,2])
        print(f'\nKey false-alarm metrics:')
        print(f'  FA AUC    : {fa_auc:.4f}')
        print(f'  FA Recall : {rec[2]:.4f}  (of {(labels==2).sum():,} FA pixels, '
              f'{int(rec[2]*(labels==2).sum()):,} correctly flagged)')
    return prec, rec, f1

prec_v, rec_v, f1_v = print_extended_report(val_preds,  val_labels,  val_probs,  'Val  (2023 — fully held-out)')
prec_t, rec_t, f1_t = print_extended_report(test_preds, test_labels, test_probs, 'Test (2018-2022 held-out 20%)')

In [ ]:
def draw_confusion_panel(ax_abs, ax_norm, preds, labels, title):
    cm_abs  = confusion_matrix(labels, preds, labels=[0,1,2])
    cm_norm = confusion_matrix(labels, preds, labels=[0,1,2], normalize='true')

    for ax, cm, is_abs in [(ax_abs, cm_abs, True), (ax_norm, cm_norm, False)]:
        ax.set_facecolor(PANEL)
        im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.max() if is_abs else 1.0)
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(CNAMES, rotation=28, ha='right', color=TEXT, fontsize=8)
        ax.set_yticklabels(CNAMES, color=TEXT, fontsize=8)
        ax.set_xlabel('Predicted', color=SUB, fontsize=9)
        ax.set_ylabel('True',      color=SUB, fontsize=9)
        sub = 'Absolute counts + row %' if is_abs else 'Row-normalised (recall view)'
        ax.set_title(f'{title}\n{sub}', color=TEXT, fontsize=9, fontweight='bold')

        for i in range(3):
            for j in range(3):
                v = cm[i, j]
                bg = v / cm.max() if is_abs else v
                fg = 'white' if bg < 0.55 else '#0D1117'
                if is_abs:
                    pct = v / cm[i].sum() * 100 if cm[i].sum() > 0 else 0
                    txt = f'{v:,}\n({pct:.1f}%)'
                else:
                    txt = f'{v:.3f}'
                    if i == j:
                        txt += '  ✓' if v >= 0.95 else ('  ⚠' if v >= 0.80 else '  ✗')
                ax.text(j, i, txt, ha='center', va='center', color=fg,
                        fontsize=8, fontweight='bold' if i==j else 'normal')
        cb = plt.colorbar(im, ax=ax, fraction=0.042, pad=0.04)
        cb.ax.yaxis.set_tick_params(color=SUB)
        plt.setp(cb.ax.yaxis.get_ticklabels(), color=SUB, fontsize=7)


fig, axes = plt.subplots(2, 2, figsize=(15, 12), facecolor=BG)
fig.patch.set_facecolor(BG)

draw_confusion_panel(axes[0][0], axes[0][1], test_preds, test_labels, 'Test (2018–2022)')
draw_confusion_panel(axes[1][0], axes[1][1], val_preds,  val_labels,  'Val (2023 holdout)')

fig.suptitle('FireSight-IR  |  Module 3b — Section C: Confusion Matrices',
             color=TEXT, fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('C_confusion_matrices.png')

In [ ]:
# ── Per-class precision / recall / F1 comparison: test vs val ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 6), facecolor=BG)
fig.patch.set_facecolor(BG)

metric_names = ['Precision', 'Recall', 'F1']
cls_colors   = [CPAL['non-fire'], CPAL['wildfire'], CPAL['false-alarm']]
x_pos        = np.arange(3)  # 3 classes
bar_w        = 0.35

for ax, (prec, rec, f1, split_name, split_tag) in zip(axes, [
    (prec_t, rec_t, f1_t, 'Test  2018–2022', 'test'),
    (prec_v, rec_v, f1_v, 'Val   2023',      'val'),
    (None, None, None, None, None),
]):
    if prec is None:
        # Third panel: combined wildfire precision/recall/F1 for both splits
        ax.set_facecolor(PANEL)
        metrics = ['Precision','Recall','F1']
        test_wf = [prec_t[1], rec_t[1], f1_t[1]]
        val_wf  = [prec_v[1], rec_v[1], f1_v[1]]
        xp = np.arange(3)
        rects1 = ax.bar(xp - bar_w/2, test_wf, bar_w, color='#4FC3F7', alpha=0.85, label='Test 2018–2022')
        rects2 = ax.bar(xp + bar_w/2, val_wf,  bar_w, color='#E85D04', alpha=0.85, label='Val 2023')
        for rects, vals in [(rects1, test_wf),(rects2, val_wf)]:
            for rect, v in zip(rects, vals):
                ax.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.005,
                        f'{v:.4f}', ha='center', va='bottom', color=TEXT, fontsize=8)
        ax.set_xticks(xp); ax.set_xticklabels(metrics, color=TEXT)
        ax.set_ylim(0, 1.12); ax.set_ylabel('Score', color=SUB)
        ax.set_title('Wildfire metrics\nTest vs Val comparison', color=TEXT,
                     fontsize=11, fontweight='bold')
        ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
        ax.grid(axis='y', alpha=0.18)
        ax.spines[['top','right']].set_visible(False)
        continue

    ax.set_facecolor(PANEL)
    for mi, (metric_arr, metric_name) in enumerate([
        ([prec[0],prec[1],prec[2]], 'Precision'),
        ([rec[0], rec[1], rec[2]],  'Recall'),
        ([f1[0],  f1[1],  f1[2]],   'F1'),
    ]):
        # Already have per-class, just need all metrics per class per split
        pass

    all_vals = np.array([[prec[0],rec[0],f1[0]],
                          [prec[1],rec[1],f1[1]],
                          [prec[2],rec[2],f1[2]]])
    xp = np.arange(3)  # metrics
    for ci, (cname, col) in enumerate(zip(CNAMES, cls_colors)):
        offset = (ci-1)*bar_w
        rects = ax.bar(xp+offset, all_vals[ci], bar_w, color=col, alpha=0.85, label=cname)
        for rect, v in zip(rects, all_vals[ci]):
            ax.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.005,
                    f'{v:.3f}', ha='center', va='bottom', color=TEXT, fontsize=7.5)
    ax.set_xticks(xp); ax.set_xticklabels(metric_names, color=TEXT)
    ax.set_ylim(0, 1.18); ax.set_ylabel('Score', color=SUB)
    ax.set_title(f'Per-class metrics\n{split_name}', color=TEXT, fontsize=11, fontweight='bold')
    ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
    ax.grid(axis='y', alpha=0.18)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('FireSight-IR  |  Module 3b — Section C: Per-class Precision / Recall / F1',
             color=TEXT, fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('C_per_class_metrics.png')

In [ ]:
# ── ROC curves: all three classes, test + val ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), facecolor=BG)
fig.patch.set_facecolor(BG)

for ax, (probs, labels, title) in zip(axes, [
    (test_probs, test_labels, 'Test (2018–2022)'),
    (val_probs,  val_labels,  'Val (2023 holdout)'),
]):
    ax.set_facecolor(PANEL)
    ax.plot([0,1],[0,1], color=BORDER, lw=1.2, ls='--', label='Random (AUC=0.50)')
    for ci, (cname, col) in enumerate(zip(
        ['Non-fire','Wildfire','False-alarm'],
        [CPAL['non-fire'], CPAL['wildfire'], CPAL['false-alarm']]
    )):
        bl  = (labels == ci).astype(int)
        if bl.sum() == 0: continue
        fpr, tpr, thresh = roc_curve(bl, probs[:, ci])
        auc = roc_auc_score(bl, probs[:, ci])
        ax.plot(fpr, tpr, color=col, lw=2.2,
                label=f'{cname}  AUC = {auc:.4f}')
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
    ax.set_xlabel('False Positive Rate', color=SUB, fontsize=10)
    ax.set_ylabel('True Positive Rate',  color=SUB, fontsize=10)
    ax.set_title(title, color=TEXT, fontsize=12, fontweight='bold')
    ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
    ax.grid(alpha=0.15)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('FireSight-IR  |  Module 3b — Section C: ROC Curves',
             color=TEXT, fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('C_roc_curves.png')

---
## Section D — Ablation Results Visualisation

> **Run this section after all three ablation variants have finished.**  
> If they completed in earlier sessions, the saved metrics are loaded automatically.


In [ ]:
# ── Load ablation results (handles case where this runs in a fresh session) ───
all_results_path = f'{ABLATION_DIR}/all_ablation_results.json'

if os.path.exists(all_results_path):
    with open(all_results_path) as f:
        ablation_results = json.load(f)
    print(f'Loaded {len(ablation_results)} variants from {all_results_path}')
else:
    # Try loading individual metric files
    ablation_results = [full_val_metrics]
    for slug, vname in [
        ('no_physics_loss',  'No physics loss'),
        ('no_era5_atm',      'No ERA5 (ATM)'),
        ('no_surface_srf',   'No surface (SRF)'),
    ]:
        mp = f'{ABLATION_DIR}/{slug}_metrics.json'
        if os.path.exists(mp):
            with open(mp) as f: ablation_results.append(json.load(f))
            print(f'  Loaded: {vname}')
        else:
            print(f'  [MISSING] {vname} — run Section B cells first')

variants = [r['variant'] for r in ablation_results]
print(f'\nVariants available: {variants}')

In [ ]:
# ── D1: Ablation metric comparison bar chart ──────────────────────────────────
metric_groups = [
    ('wf_recall',    'WF Recall',    '#E85D04'),
    ('wf_precision', 'WF Precision', '#FF8C42'),
    ('wf_f1',        'WF F1',        '#FFC080'),
    ('fa_recall',    'FA Recall',    '#D97706'),
    ('fa_auc',       'FA AUC',       '#4CAF50'),
    ('val_acc',      'Overall Acc',  '#4FC3F7'),
]

n_variants = len(ablation_results)
n_metrics  = len(metric_groups)
x          = np.arange(n_metrics)
bar_w      = 0.18

fig, ax = plt.subplots(figsize=(17, 7), facecolor=BG)
ax.set_facecolor(PANEL)

var_colors = [APAL.get(v, '#888888') for v in variants]

for vi, (r, col) in enumerate(zip(ablation_results, var_colors)):
    vals   = [r[mk] for mk, _, _ in metric_groups]
    offset = (vi - (n_variants-1)/2) * bar_w
    rects  = ax.bar(x + offset, vals, bar_w, color=col,
                    alpha=0.88 if vi == 0 else 0.78,
                    label=r['variant'],
                    edgecolor='white' if vi==0 else 'none',
                    linewidth=0.8)
    for rect, v in zip(rects, vals):
        ax.text(rect.get_x()+rect.get_width()/2,
                rect.get_height()+0.004,
                f'{v:.3f}', ha='center', va='bottom',
                color=TEXT, fontsize=6.5, rotation=55)

ax.set_xticks(x)
ax.set_xticklabels([lbl for _, lbl, _ in metric_groups], color=TEXT, fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score (Val 2023)', color=SUB, fontsize=11)
ax.set_title('Ablation Study — Component Impact on Val 2023 Metrics\nFull model vs 3 stripped variants (30 epochs each)',
             color=TEXT, fontsize=13, fontweight='bold')
ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=9,
          loc='upper right', framealpha=0.9)
ax.grid(axis='y', alpha=0.18)
ax.spines[['top','right']].set_visible(False)

# Annotate full-model bars with star
ax.text(0.01, 0.98, '★ = Full model baseline', transform=ax.transAxes,
        color=APAL['Full model'], fontsize=9, va='top')

plt.tight_layout()
savefig('D_ablation_comparison.png')

In [ ]:
# ── D2: Delta heatmap — degradation from full model ───────────────────────────
base_r      = ablation_results[0]
metric_keys = ['wf_recall','wf_precision','wf_f1','fa_recall','fa_auc','val_acc']
metric_lbls = ['WF Recall','WF Precision','WF F1','FA Recall','FA AUC','Overall Acc']
var_lbls    = [r['variant'] for r in ablation_results[1:]]

delta_matrix = np.array([
    [r[k] - base_r[k] for k in metric_keys]
    for r in ablation_results[1:]
])

fig, ax = plt.subplots(figsize=(12, 4.5), facecolor=BG)
ax.set_facecolor(PANEL)

vabs = max(abs(delta_matrix.min()), abs(delta_matrix.max()), 0.01)
im   = ax.imshow(delta_matrix, cmap='RdYlGn', vmin=-vabs, vmax=vabs, aspect='auto')

ax.set_xticks(range(len(metric_keys)))
ax.set_yticks(range(len(var_lbls)))
ax.set_xticklabels(metric_lbls, color=TEXT, fontsize=10)
ax.set_yticklabels(var_lbls, color=TEXT, fontsize=10)

for i in range(len(var_lbls)):
    for j in range(len(metric_keys)):
        v   = delta_matrix[i, j]
        txt = f'{v:+.4f}'
        ax.text(j, i, txt, ha='center', va='center',
                color='#0D1117' if abs(v) < vabs*0.5 else 'white',
                fontsize=9, fontweight='bold')

cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.ax.yaxis.set_tick_params(color=SUB)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=SUB)
cb.set_label('Delta from full model', color=SUB)

ax.set_title('Ablation Δ heatmap  (green = no degradation, red = degradation)',
             color=TEXT, fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('D_ablation_delta_heatmap.png')

In [ ]:
# ── D3: Convergence curves for all variants ───────────────────────────────────
# Load training logs for each variant

log_sources = {
    'Full model'       : f'{MODEL_DIR}/training_log.json',
    'No physics loss'  : f'{ABLATION_DIR}/no_physics_loss_log.json',
    'No ERA5 (ATM)'    : f'{ABLATION_DIR}/no_era5_atm_log.json',
    'No surface (SRF)' : f'{ABLATION_DIR}/no_surface_srf_log.json',
}

logs = {}
for name, path in log_sources.items():
    if os.path.exists(path):
        with open(path) as f: logs[name] = json.load(f)
        print(f'  Loaded {len(logs[name])} epochs: {name}')
    else:
        print(f'  [MISSING] {name} log — variant may not have run yet')

if len(logs) < 2:
    print('\n[INFO] Run ablation variants (Section B) before plotting convergence.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor=BG)
    fig.patch.set_facecolor(BG)

    plot_specs = [
        ('val_loss',                    'Val Loss',          False),
        ('val_acc',                     'Val Accuracy',      True),
        ('per_class_val_acc.wf',        'WF Val Recall',     True),
        ('per_class_val_acc.fa',        'FA Val Recall',     True),
        ('loss_components.ce',          'CE Loss component', False),
        ('loss_components.bl',          'BL Loss component', False),
    ]

    def get_series(log, key):
        parts = key.split('.')
        if len(parts) == 1:
            return [e.get(parts[0], np.nan) for e in log]
        return [e.get(parts[0], {}).get(parts[1], np.nan) for e in log]

    for ax, (key, title, higher_better) in zip(axes.flat, plot_specs):
        ax.set_facecolor(PANEL)
        for name, log in logs.items():
            ep   = [e['epoch'] for e in log]
            vals = get_series(log, key)
            col  = APAL.get(name, '#888888')
            lw   = 2.4 if name == 'Full model' else 1.8
            ls   = '-'  if name == 'Full model' else '--'
            ax.plot(ep, vals, color=col, lw=lw, ls=ls, label=name, alpha=0.9)
        ax.set_xlabel('Epoch', color=SUB, fontsize=9)
        ax.set_ylabel(title,   color=SUB, fontsize=9)
        ax.set_title(title, color=TEXT, fontsize=10, fontweight='bold')
        ax.legend(facecolor=PANEL2, edgecolor=BORDER, labelcolor=TEXT, fontsize=7.5)
        ax.grid(alpha=0.16)
        ax.spines[['top','right']].set_visible(False)

    fig.suptitle('FireSight-IR  |  Module 3b — Section D: Ablation Convergence Curves',
                 color=TEXT, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    savefig('D_ablation_convergence.png')

---
## Summary

In [ ]:
print('=' * 65)
print('  FireSight-IR  |  Module 3b — Analysis & Ablation — Summary')
print('=' * 65)

outputs = [
    (f'{FIGURES_DIR}/A_score_distributions.png',    'A  Probability score distributions'),
    (f'{FIGURES_DIR}/C_confusion_matrices.png',     'C  Confusion matrices (counts + normalised)'),
    (f'{FIGURES_DIR}/C_per_class_metrics.png',      'C  Per-class precision / recall / F1'),
    (f'{FIGURES_DIR}/C_roc_curves.png',             'C  ROC curves — test and val'),
    (f'{FIGURES_DIR}/D_ablation_comparison.png',    'D  Ablation metric comparison'),
    (f'{FIGURES_DIR}/D_ablation_delta_heatmap.png', 'D  Ablation delta heatmap'),
    (f'{FIGURES_DIR}/D_ablation_convergence.png',   'D  Ablation convergence curves'),
    (f'{ABLATION_DIR}/all_ablation_results.json',   'B  Ablation results JSON'),
]

for path, desc in outputs:
    e = '✓' if os.path.exists(path) else '○ (pending)'
    print(f'  {e}  {desc}')

print()
print('  Full model Val 2023 summary:')
print(f'    WF recall     : {full_val_metrics["wf_recall"]:.4f}')
print(f'    WF precision  : {full_val_metrics["wf_precision"]:.4f}')
print(f'    FA recall     : {full_val_metrics["fa_recall"]:.4f}')
print(f'    FA AUC        : {full_val_metrics["fa_auc"]:.4f}')
print(f'    Overall acc   : {full_val_metrics["val_acc"]:.4f}')
print('=' * 65)